# MergeDeclaredPreprints — nightly preprint ↔ article merges (oxjob #1099)

Runs between `Locations_with_Types` and `Map_Work_Ids`. MapWorkIds' `crossref_preprint` tier joins a record to its declared
partner only when the link is there on the record's first night; most links are added later (the preprint server records the
article after it is out, and Crossref mirrors the claim onto the article as `has-preprint`), when both records already sit on
their own works and are pinned. This step merges those works every night (Casey 2026-10-08: "build it into the nightly").

- **Pairs** (view `openalex.works.preprint_link_pairs`, recreated here first; MapWorkIds' `crossref_preprint` tier reads its
  Crossref arm): declared by either Crossref record (`is-preprint-of` / `has-preprint` in `ids[]`), or by arXiv's DataCite record
  (`IsVersionOf` a journal DOI; arXiv adds it after publication, oxjob #1256's declared_version pairs); preprint = Crossref
  posted-content/preprint or a DataCite `preprint` (arXiv 10.48550), article = Crossref journal-article typed article / review /
  letter; both pinned, on two works that are in `openalex_works`.
- **Group** = an article's work + the works of every preprint it pairs with. **Winner = the most-cited member, ties to the older
  (lower) id.** A just-arrived article has no citations yet, so it joins the preprint's work (Casey 2026-10-08: the published work
  that comes in and points to a preprint just joins the preprint work); an established, cited article keeps its id (first-night
  backlog: the T2T genome, gnomAD, ARRIVE 2.0 articles would otherwise 404 onto their bioRxiv ids). The merged work presents
  as the article either way (CreateWorksBase prefers the published Crossref VoR, #589).
- **Holds** (evidence of a bad link): the article's work titled as a correction, typed other than article / review / letter, a
  year gap (article > 3 years after or > 1 before the preprint), or content that differs (titles differ and abstract token Jaccard
  < 0.6 or no abstract to compare: blind eval 2026-10-09, 0/100 wrong where content agrees; the wrong links all differ); arXiv-only:
  the preprint's work already shows another DOI. Mechanical: a preprint work paired with two article works, a work on both sides, a
  loser key held by a work outside the group. Any held pair holds its whole group.
- **Execute** (as `MergeIdenticalKeyWorks` execute): audit → aliases of the losers' record keys onto the winner → drop the losers'
  map rows → unpin the losers → `merged_work_ids` → ledger. MapWorkIds then re-resolves the losers' records onto the winner in this
  run. Map rows go before pins, so a partial run leaves a loser's records pinned where they were, never re-resolving onto it.
- **Citations:** `work_references` rows citing a ledgered loser, and the loser's own reference rows, move to the winner.
- **Tripwire:** more than 100,000 executable losers in one night fails the task before any write (a bug, not a backlog: the
  first night's backlog was ~10K). `Merge_Declared_Preprints_Gate` passes either way, so MapWorkIds always runs.
- **Undo** (before the next nightly): re-insert `preprint_link_merges_audit` pin/map rows of the run, delete its alias rows from
  `work_id_map`, delete the run's `merged_work_ids` rows (source `preprint_link_nightly`).

In [ ]:
CREATE OR REPLACE VIEW openalex.works.preprint_link_pairs
COMMENT 'oxjob #1099: declared preprint <-> published-article DOI pairs, typed. One definition for MapWorkIds (crossref_preprint tier, arm = crossref) and MergeDeclaredPreprints (both arms). Recreated nightly by MergeDeclaredPreprints.'
AS
-- Crossref: declared by either record (`is-preprint-of` on the preprint, `has-preprint` on the article; Crossref mirrors one
-- side's claim onto the other)
WITH cr_links AS (
  SELECT DISTINCT
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') END AS pre,
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') END AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'crossref' AND EXISTS(l.ids, y -> y.relationship IN ('is-preprint-of', 'has-preprint'))
    AND x.relationship IN ('is-preprint-of', 'has-preprint')
),
-- arXiv's DataCite record declaring IsVersionOf its journal DOI (oxjob #1256's declared_version pairs); arXiv adds the DOI
-- after the article is out, so most of these appear when both records are already pinned
dc_links AS (
  SELECT DISTINCT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS pre,
         regexp_replace(regexp_replace(LOWER(x.id), '^https?://(dx\\.)?doi\\.org/', ''), '[^a-zA-Z0-9\./-]', '') AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'datacite' AND l.type = 'preprint' AND l.native_id LIKE '10.48550/%'
    AND x.relationship = 'IsVersionOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND LOWER(x.id) NOT LIKE '%10.48550/%'
),
links AS (
  SELECT pre, pub, CASE WHEN BOOL_OR(arm = 'crossref') THEN 'crossref' ELSE 'datacite' END AS arm
  FROM (SELECT pre, pub, 'crossref' AS arm FROM cr_links UNION ALL SELECT pre, pub, 'datacite' FROM dc_links)
  WHERE pre <> pub
  GROUP BY pre, pub
)
-- typed: preprint = Crossref posted-content/preprint or a DataCite preprint (AACR registers ~204K Figshare supplements as
-- `is-preprint-of` with subtype `other`); article = Crossref journal-article typed article / review / letter
SELECT k.pre, k.pub, k.arm FROM links k
LEFT SEMI JOIN openalex.works.locations_w_types a
  ON a.provenance IN ('crossref', 'datacite') AND regexp_replace(LOWER(a.native_id), '[^a-zA-Z0-9\./-]', '') = k.pre
 AND ((a.provenance = 'crossref' AND a.raw_type = 'posted-content' AND a.type = 'preprint')
   OR (a.provenance = 'datacite' AND a.type = 'preprint'))
LEFT SEMI JOIN openalex.works.locations_w_types b
  ON b.provenance = 'crossref' AND regexp_replace(b.native_id, '[^a-zA-Z0-9\./-]', '') = k.pub
 AND b.raw_type = 'journal-article' AND b.type IN ('article', 'review', 'letter')

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.preprint_link_merges (
  run_ts TIMESTAMP, loser_work_id BIGINT, winner_work_id BIGINT, group_work_id BIGINT, n_pairs INT, signals STRING)
CLUSTER BY (loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.preprint_link_merges_audit (
  run_ts TIMESTAMP, kind STRING, loser_work_id BIGINT, winner_work_id BIGINT,
  provenance STRING, native_id_namespace STRING, native_id STRING, work_id_source STRING,
  openalex_created_dt DATE, openalex_updated_dt TIMESTAMP, seeded_dt DATE, seeded_from STRING,
  doi STRING, pmid STRING, arxiv STRING, title_author STRING, created_date DATE, updated_date TIMESTAMP)
CLUSTER BY (run_ts, loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.preprint_link_merges_refs_audit (
  run_ts TIMESTAMP, side STRING, citing_work_id BIGINT, native_id STRING, native_id_namespace STRING, ref_ind BIGINT,
  old_id BIGINT, new_id BIGINT)

## Pairs on two works, with the evidence holds

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_pairs AS
WITH links AS (SELECT pre, pub, arm = 'crossref' AS cr_declared FROM openalex.works.preprint_link_pairs),
ld AS (SELECT pre AS d FROM links UNION SELECT pub FROM links),
lt AS (
  SELECT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, l.provenance, r.work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ld ON ld.d = regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '')
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance IN ('crossref', 'datacite') AND r.work_id IS NOT NULL
),
pw AS (
  SELECT DISTINCT k.pre AS pre_doi, k.pub AS pub_doi, k.cr_declared, a.work_id AS pre_work, b.work_id AS pub_work
  FROM links k JOIN lt a ON a.d = k.pre JOIN lt b ON b.d = k.pub AND b.provenance = 'crossref'
  WHERE a.work_id <> b.work_id
),
wp AS (SELECT DISTINCT pre_work, pub_work FROM pw),
ids AS (SELECT pre_work AS id FROM wp UNION SELECT pub_work FROM wp),
wsel AS (
  SELECT w.id, w.publication_year AS yr, w.type, w.title, COALESCE(w.cited_by_count, 0) AS cites, w.abstract,
         regexp_replace(lower(w.title), '[^\\p{L}\\p{N}]', '') AS tn,
         regexp_replace(lower(w.doi), '^https?://(dx\\.)?doi\\.org/', '') AS doi
  FROM openalex.works.openalex_works w LEFT SEMI JOIN ids ON ids.id = w.id
),
-- abstract tokens as rows, Jaccard by join: array set functions over per-work token lists gave run-varying answers in both
-- directions on the warehouse (2026-10-07 arrays_overlap; 2026-10-09 size(array_intersect): 61-193 pairs flipped per run)
atok AS (
  SELECT DISTINCT id, t FROM wsel
  LATERAL VIEW explode(split(regexp_replace(lower(COALESCE(abstract, '')), '[^a-z0-9 ]', ' '), ' +')) e AS t
  WHERE length(t) > 3
),
nab AS (SELECT id, COUNT(*) AS n FROM atok GROUP BY id),
ain AS (SELECT p.pre_work, p.pub_work, COUNT(*) AS inter FROM wp p JOIN atok a ON a.id = p.pre_work JOIN atok b ON b.id = p.pub_work AND b.t = a.t GROUP BY 1, 2),
wf AS (
  SELECT s.id, s.yr, s.type, s.title, s.cites, s.tn, s.doi, COALESCE(na.n, 0) AS n_abs
  FROM wsel s LEFT JOIN nab na ON na.id = s.id
),
r AS (
  SELECT p.*, x.yr AS pre_yr, y.yr AS pub_yr, y.type AS pub_type, x.cites AS pre_cites, y.cites AS pub_cites,
         LEFT(x.title, 200) AS pre_title, LEFT(y.title, 200) AS pub_title, y.title AS pub_title_full, x.tn = y.tn AS title_same,
         x.doi AS pre_work_doi,
         CASE WHEN x.n_abs >= 20 AND y.n_abs >= 20 THEN COALESCE(ai.inter, 0) / (x.n_abs + y.n_abs - COALESCE(ai.inter, 0)) END AS abs_jaccard
  FROM pw p JOIN wf x ON x.id = p.pre_work JOIN wf y ON y.id = p.pub_work
  LEFT JOIN ain ai ON ai.pre_work = p.pre_work AND ai.pub_work = p.pub_work
)
SELECT pre_doi, pub_doi, pre_work, pub_work, pre_yr, pub_yr, pub_type, pre_cites, pub_cites, pre_title, pub_title,
       CASE WHEN cr_declared THEN 'crossref' ELSE 'datacite' END AS arm, abs_jaccard,
       concat_ws('+', 'declared', CASE WHEN title_same THEN 'title' END, CASE WHEN abs_jaccard >= 0.6 THEN 'abstract' END) AS signals,
       CASE WHEN pub_title_full RLIKE '(?i)^\\s*(metadata correction|correction|erratum|corrigendum|addendum)([^a-z]|$)' THEN 'correction_notice'
            WHEN pub_type NOT IN ('article', 'review', 'letter') THEN 'article_type'
            WHEN pre_yr IS NOT NULL AND pub_yr IS NOT NULL AND (pub_yr - pre_yr > 3 OR pub_yr - pre_yr < -1) THEN 'year_gap'
            -- the declaration merges when the content agrees: same normalized title, or abstracts agree (token Jaccard >= 0.6). Blind
            -- eval 2026-10-09 (240 pairs): 0/100 wrong where content agrees; every wrong link (JMIR's 2018 off-by-N deposits, OSF
            -- REMOVED/WITHDRAWN placeholders, mistyped DOIs) had a different title and abstracts < 0.6 or none. An author-name check
            -- let ~0.7 % wrong through (common surnames, same-group papers) and held collaboration papers ("ATLAS Collaboration").
            WHEN NOT title_same AND COALESCE(abs_jaccard, 0) < 0.6 THEN 'content_differs'
            -- arXiv-only: the preprint's work already presents another DOI (a journal record glued on): a three-way case
            WHEN NOT cr_declared AND pre_work_doi IS NOT NULL AND pre_work_doi <> pre_doi AND pre_work_doi NOT LIKE '10.48550/%' THEN 'pre_work_other_doi'
       END AS evidence_hold
FROM r

## Groups, winners, mechanical holds — one row per loser

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_losers AS
WITH multi AS (SELECT pre_work FROM openalex.works.preprint_link_merge_pairs GROUP BY pre_work HAVING COUNT(DISTINCT pub_work) > 1),
both_sides AS (SELECT DISTINCT a.pub_work AS w FROM openalex.works.preprint_link_merge_pairs a
               JOIN openalex.works.preprint_link_merge_pairs b ON b.pre_work = a.pub_work),
pr AS (
  SELECT p.*,
         COALESCE(p.evidence_hold,
                  CASE WHEN m.pre_work IS NOT NULL THEN 'multi_article'
                       WHEN c1.w IS NOT NULL OR c2.w IS NOT NULL THEN 'chained' END) AS pair_hold
  FROM openalex.works.preprint_link_merge_pairs p
  LEFT JOIN multi m ON m.pre_work = p.pre_work
  LEFT JOIN both_sides c1 ON c1.w = p.pub_work
  LEFT JOIN both_sides c2 ON c2.w = p.pre_work
),
mc AS (SELECT pub_work AS g, pre_work AS w, pre_cites AS cites FROM pr UNION SELECT pub_work, pub_work, pub_cites FROM pr),
grp AS (
  SELECT p.pub_work AS group_work_id, w.winner_work_id, COUNT(*) AS n_pairs, MIN(p.pair_hold) AS group_hold, MIN(p.signals) AS signals
  FROM pr p
  JOIN (SELECT g, MAX_BY(w, named_struct('cites', cites, 'older', -w)) AS winner_work_id FROM mc GROUP BY g) w ON w.g = p.pub_work
  GROUP BY p.pub_work, w.winner_work_id
),
members AS (
  SELECT DISTINCT g.group_work_id, g.winner_work_id, m.w AS work_id
  FROM grp g JOIN mc m ON m.g = g.group_work_id
),
losers AS (
  SELECT m.group_work_id, m.winner_work_id, m.work_id AS loser_work_id, g.n_pairs, g.signals, g.group_hold
  FROM members m JOIN grp g ON g.group_work_id = m.group_work_id
  WHERE m.work_id <> m.winner_work_id
),
-- the losers' record keys, as MapWorkIds will look them up after the unpin
lrec AS (
  SELECT lo.group_work_id, lo.loser_work_id, l.merge_key AS mk
  FROM losers lo
  JOIN openalex.works.location_work_ids r ON r.work_id = lo.loser_work_id
  JOIN openalex.works.locations_w_types l
    ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
  WHERE lo.group_hold IS NULL
),
lkeys AS (
  SELECT DISTINCT x.group_work_id, x.loser_work_id, kv.col, kv.k
  FROM lrec x LATERAL VIEW explode(map('doi', NULLIF(x.mk.doi, ''), 'pmid', NULLIF(x.mk.pmid, ''),
                                       'arxiv', CASE WHEN x.mk.arxiv IN ('arXiv:', '') THEN NULL ELSE x.mk.arxiv END,
                                       'title_author', NULLIF(x.mk.title_author, ''))) kv AS col, k
  WHERE kv.k IS NOT NULL
),
keyless AS (
  SELECT DISTINCT group_work_id, loser_work_id FROM lrec
  WHERE NULLIF(mk.doi, '') IS NULL AND NULLIF(mk.pmid, '') IS NULL AND (mk.arxiv IS NULL OR mk.arxiv IN ('arXiv:', ''))
),
kin AS (SELECT DISTINCT col, k FROM lkeys),
holders AS (
  SELECT 'doi' AS col, m.doi AS k, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'doi' AND kin.k = m.doi
  UNION ALL SELECT 'pmid', m.pmid, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'pmid' AND kin.k = m.pmid
  UNION ALL SELECT 'arxiv', m.arxiv, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'arxiv' AND kin.k = m.arxiv
  UNION ALL SELECT 'title_author', m.title_author, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'title_author' AND kin.k = m.title_author
),
-- a key held by a work outside the group; a shared title key only counts for a loser with a doi / pmid / arxiv-less record
shared AS (
  SELECT DISTINCT lk.group_work_id FROM lkeys lk JOIN holders h ON h.col = lk.col AND h.k = lk.k
  WHERE NOT EXISTS (SELECT 1 FROM members m WHERE m.group_work_id = lk.group_work_id AND m.work_id = h.id)
    AND (lk.col <> 'title_author' OR EXISTS (SELECT 1 FROM keyless kl WHERE kl.group_work_id = lk.group_work_id AND kl.loser_work_id = lk.loser_work_id))
)
SELECT lo.group_work_id, lo.winner_work_id, lo.loser_work_id, lo.n_pairs, lo.signals,
       COALESCE(lo.group_hold, CASE WHEN sh.group_work_id IS NOT NULL THEN 'loser_key_shared' END) AS hold_reason,
       current_timestamp() AS run_ts
FROM losers lo LEFT JOIN shared sh ON sh.group_work_id = lo.group_work_id

In [ ]:
SELECT hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.preprint_link_merge_losers GROUP BY 1 ORDER BY 2 DESC

## Tripwire and invariants (before any write)

In [ ]:
WITH x AS (SELECT * FROM openalex.works.preprint_link_merge_losers WHERE hold_reason IS NULL),
s AS (SELECT COUNT(*) AS n, COUNT(DISTINCT loser_work_id) AS n_losers FROM x),
c AS (SELECT COUNT(*) AS loser_is_winner FROM x JOIN (SELECT DISTINCT winner_work_id FROM x) w ON w.winner_work_id = x.loser_work_id)
SELECT CASE
  WHEN s.n > 100000 THEN RAISE_ERROR(concat('MergeDeclaredPreprints: ', s.n, ' executable losers > 100,000 tripwire; nothing written'))
  WHEN s.n_losers <> s.n THEN RAISE_ERROR('MergeDeclaredPreprints: a loser has two winners')
  WHEN c.loser_is_winner > 0 THEN RAISE_ERROR('MergeDeclaredPreprints: a loser is also a winner')
END AS tripwire
FROM s CROSS JOIN c

## Execute
Audit first, aliases before the unpin (the aliases are read from the losers' pinned records), map rows before pins.

In [ ]:
INSERT INTO openalex.works.preprint_link_merges_audit
SELECT x.run_ts, 'pin', x.loser_work_id, x.winner_work_id,
       r.provenance, r.native_id_namespace, r.native_id, r.work_id_source, r.openalex_created_dt, r.openalex_updated_dt, r.seeded_dt, r.seeded_from,
       NULL, NULL, NULL, NULL, NULL, NULL
FROM openalex.works.preprint_link_merge_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
INSERT INTO openalex.works.preprint_link_merges_audit
SELECT x.run_ts, 'map', x.loser_work_id, x.winner_work_id,
       NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       m.doi, m.pmid, m.arxiv, m.title_author, m.created_date, m.updated_date
FROM openalex.works.preprint_link_merge_losers x
JOIN openalex.works.work_id_map m ON m.id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_aliases AS
SELECT DISTINCT x.run_ts, x.loser_work_id, x.winner_work_id AS id, NULLIF(l.merge_key.doi, '') AS doi, l.merge_key.pmid AS pmid,
       l.merge_key.arxiv AS arxiv, l.merge_key.title_author AS title_author
FROM openalex.works.preprint_link_merge_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
JOIN openalex.works.locations_w_types l
  ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
WHERE x.hold_reason IS NULL
  AND COALESCE(NULLIF(l.merge_key.doi, ''), l.merge_key.pmid, l.merge_key.arxiv, l.merge_key.title_author) IS NOT NULL

In [ ]:
INSERT INTO openalex.works.work_id_map (id, doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT DISTINCT id, doi, pmid, arxiv, title_author, current_date(), current_timestamp()
FROM openalex.works.preprint_link_merge_aliases x
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.work_id_map m WHERE m.id = x.id AND m.doi <=> x.doi AND m.pmid <=> x.pmid
                  AND m.arxiv <=> x.arxiv AND m.title_author <=> x.title_author)

In [ ]:
INSERT INTO openalex.works.preprint_link_merges_audit
SELECT run_ts, 'alias', loser_work_id, id, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       doi, pmid, arxiv, title_author, current_date(), NULL
FROM openalex.works.preprint_link_merge_aliases

In [ ]:
DELETE FROM openalex.works.work_id_map m
WHERE EXISTS (SELECT 1 FROM openalex.works.preprint_link_merge_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = m.id)

In [ ]:
DELETE FROM openalex.works.location_work_ids r
WHERE EXISTS (SELECT 1 FROM openalex.works.preprint_link_merge_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = r.work_id)

In [ ]:
MERGE INTO openalex.works.merged_work_ids m
USING (SELECT DISTINCT loser_work_id, winner_work_id, run_ts FROM openalex.works.preprint_link_merge_losers WHERE hold_reason IS NULL) s
ON m.loser_work_id = s.loser_work_id AND m.winner_work_id = s.winner_work_id
WHEN NOT MATCHED THEN INSERT (loser_work_id, winner_work_id, merged_at, source)
VALUES (s.loser_work_id, s.winner_work_id, s.run_ts, 'preprint_link_nightly')

In [ ]:
INSERT INTO openalex.works.preprint_link_merges
SELECT run_ts, loser_work_id, winner_work_id, group_work_id, n_pairs, signals
FROM openalex.works.preprint_link_merge_losers WHERE hold_reason IS NULL

## Citations follow the merge
Rows citing a ledgered loser move to its winner, and the loser's own reference rows come along (they keep their
`native_id` + `ref_ind`, so they never collide with the winner's). Idempotent: only rows still keyed to a loser move.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW preprint_link_repoint AS
SELECT p.loser_work_id, p.winner_work_id
FROM (SELECT loser_work_id, MIN(winner_work_id) AS winner_work_id FROM openalex.works.preprint_link_merges GROUP BY loser_work_id) p
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.location_work_ids r WHERE r.work_id = p.loser_work_id)

In [ ]:
INSERT INTO openalex.works.preprint_link_merges_refs_audit
SELECT current_timestamp(), 'cited', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.cited_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN preprint_link_repoint p ON r.cited_work_id = p.loser_work_id
UNION ALL
SELECT current_timestamp(), 'citing', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.citing_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN preprint_link_repoint p ON r.citing_work_id = p.loser_work_id

In [ ]:
MERGE INTO openalex.works.work_references r
USING preprint_link_repoint p ON r.cited_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.cited_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
MERGE INTO openalex.works.work_references r
USING preprint_link_repoint p ON r.citing_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.citing_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
SELECT (SELECT COUNT(*) FROM openalex.works.preprint_link_merge_pairs) AS pairs_on_two_works,
       (SELECT COUNT_IF(hold_reason IS NULL) FROM openalex.works.preprint_link_merge_losers) AS losers_merged,
       (SELECT COUNT(DISTINCT winner_work_id) FROM openalex.works.preprint_link_merge_losers WHERE hold_reason IS NULL) AS winners,
       (SELECT COUNT(*) FROM openalex.works.preprint_link_merge_aliases) AS alias_rows,
       (SELECT COUNT(*) FROM openalex.works.preprint_link_merges) AS losers_merged_all_time